# TP2 - Sistema de Recomendación de Anime
## Análisis Descriptivo

**Dataset:** [Anime Recommendations Database - Kaggle](https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database)

**Archivos:**
- `data/anime.csv`: metadata de 12,294 animes
- `data/rating.parquet`: 7.8M ratings de usuarios

**Objetivo del análisis:** entender la estructura del dataset antes de construir el sistema de recomendación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Carga de datos

In [ ]:
anime = pd.read_csv('../data/anime.csv')
ratings = pd.read_parquet('../data/rating.parquet')

print(f'anime.csv:          {anime.shape[0]:,} filas x {anime.shape[1]} columnas')
print(f'rating.parquet:     {ratings.shape[0]:,} filas x {ratings.shape[1]} columnas')

---
## 2. Exploración inicial

In [ ]:
# Vista general del dataset de animes
anime.head(10)

In [ ]:
# Vista general del dataset de ratings
ratings.head(10)

In [ ]:
# Tipos de datos y nulos
print('=== anime.csv ===')
print(anime.dtypes)
print()
print('Valores nulos:')
print(anime.isnull().sum())

In [ ]:
print('=== rating.parquet ===')
print(ratings.dtypes)
print()
print('Valores nulos:')
print(ratings.isnull().sum())

In [ ]:
# Estadísticas descriptivas
anime.describe()

In [ ]:
ratings.describe()

---
## 3. Análisis del dataset de Animes (`anime.csv`)

### 3.1 Distribución por tipo

In [ ]:
type_counts = anime['type'].value_counts(dropna=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
sns.barplot(x=type_counts.index.astype(str), y=type_counts.values, ax=axes[0], palette='Blues_d')
axes[0].set_title('Cantidad de animes por tipo')
axes[0].set_xlabel('Tipo')
axes[0].set_ylabel('Cantidad')
for bar, val in zip(axes[0].patches, type_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f'{val:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(type_counts.values, labels=type_counts.index.astype(str), autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporción por tipo')

plt.tight_layout()
plt.show()

print(type_counts)

### 3.2 Distribución del rating promedio de animes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
anime['rating'].dropna().hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución del rating promedio de animes')
axes[0].set_xlabel('Rating promedio')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(anime['rating'].mean(), color='red', linestyle='--', label=f"Media: {anime['rating'].mean():.2f}")
axes[0].axvline(anime['rating'].median(), color='orange', linestyle='--', label=f"Mediana: {anime['rating'].median():.2f}")
axes[0].legend()

# Boxplot por tipo
anime_notna = anime.dropna(subset=['rating', 'type'])
order = anime_notna.groupby('type')['rating'].median().sort_values(ascending=False).index
sns.boxplot(data=anime_notna, x='type', y='rating', order=order, ax=axes[1], palette='Blues')
axes[1].set_title('Rating promedio por tipo de anime')
axes[1].set_xlabel('Tipo')
axes[1].set_ylabel('Rating')

plt.tight_layout()
plt.show()

### 3.3 Géneros más frecuentes

In [ ]:
# Cada anime puede tener múltiples géneros separados por coma
all_genres = anime['genre'].dropna().str.split(', ').explode()
genre_counts = all_genres.value_counts().head(20)

plt.figure(figsize=(14, 6))
sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='Blues_r')
plt.title('Top 20 géneros más frecuentes')
plt.xlabel('Cantidad de animes')
plt.ylabel('Género')
for i, val in enumerate(genre_counts.values):
    plt.text(val + 10, i, f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f'Total de géneros únicos: {all_genres.nunique()}')

### 3.4 Cantidad de géneros por anime

In [ ]:
anime['n_genres'] = anime['genre'].dropna().str.split(', ').apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

anime['n_genres'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Cantidad de géneros por anime')
axes[0].set_xlabel('Número de géneros')
axes[0].set_ylabel('Cantidad de animes')
axes[0].tick_params(axis='x', rotation=0)

# Distribución de miembros (popularidad)
np.log1p(anime['members']).hist(bins=40, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de popularidad (log de miembros)')
axes[1].set_xlabel('log(1 + miembros)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### 3.5 Top animes por popularidad y por rating

In [ ]:
# Top 15 por cantidad de miembros (popularidad)
top_popular = anime.nlargest(15, 'members')[['name', 'type', 'rating', 'members']]

plt.figure(figsize=(14, 6))
sns.barplot(x='members', y='name', data=top_popular, palette='Blues_r')
plt.title('Top 15 animes más populares (por miembros)')
plt.xlabel('Miembros')
plt.ylabel('')
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 por rating promedio (con al menos 1000 miembros para filtrar animes oscuros)
top_rated = anime[anime['members'] >= 1000].nlargest(15, 'rating')[['name', 'type', 'rating', 'members']]

plt.figure(figsize=(14, 6))
bars = sns.barplot(x='rating', y='name', data=top_rated, palette='Greens_r')
plt.title('Top 15 animes mejor rankeados (mín. 1000 miembros)')
plt.xlabel('Rating promedio')
plt.ylabel('')
plt.xlim(8, 10)
plt.tight_layout()
plt.show()

---
## 4. Análisis del dataset de Ratings (`rating.parquet`)

### 4.1 Ratings explícitos vs implícitos

El dataset tiene dos tipos de interacciones:
- **Rating = -1**: el usuario vio el anime pero **no lo rateó** (interacción implícita)
- **Rating 1-10**: el usuario vio el anime y le **asignó un puntaje** (interacción explícita)

In [ ]:
n_implicitos = (ratings['rating'] == -1).sum()
n_explicitos = (ratings['rating'] != -1).sum()
total = len(ratings)

print(f'Total de interacciones:      {total:,}')
print(f'Ratings explícitos (1-10):   {n_explicitos:,} ({n_explicitos/total*100:.1f}%)')
print(f'Ratings implícitos (-1):     {n_implicitos:,} ({n_implicitos/total*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie
axes[0].pie([n_explicitos, n_implicitos], labels=['Explícito (1-10)', 'Implícito (-1)'],
            autopct='%1.1f%%', colors=['steelblue', 'lightcoral'])
axes[0].set_title('Proporción de ratings explícitos vs implícitos')

# Distribución de ratings explícitos
ratings_exp = ratings[ratings['rating'] != -1]['rating']
ratings_exp.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de ratings explícitos (1-10)')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Cantidad')
axes[1].tick_params(axis='x', rotation=0)
axes[1].axvline(ratings_exp.mean() - 1, color='red', linestyle='--', label=f'Media: {ratings_exp.mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.2 Actividad por usuario

In [ ]:
ratings_por_usuario = ratings.groupby('user_id')['rating'].count()

print('Estadísticas de ratings por usuario:')
print(ratings_por_usuario.describe().apply(lambda x: f'{x:,.1f}'))
print(f'\nUsuarios únicos: {ratings["user_id"].nunique():,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución (escala log)
np.log1p(ratings_por_usuario).hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de ratings por usuario (log scale)')
axes[0].set_xlabel('log(1 + cantidad de ratings)')
axes[0].set_ylabel('Cantidad de usuarios')

# Usuarios con pocos ratings (problema cold-start)
umbrales = [1, 5, 10, 20, 50]
porcentajes = [(ratings_por_usuario <= u).mean() * 100 for u in umbrales]
axes[1].bar([str(u) for u in umbrales], porcentajes, color='lightcoral', edgecolor='white')
axes[1].set_title('% de usuarios con N o menos ratings (cold-start)')
axes[1].set_xlabel('Umbral de ratings')
axes[1].set_ylabel('% de usuarios')
for i, p in enumerate(porcentajes):
    axes[1].text(i, p + 0.5, f'{p:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

### 4.3 Cobertura por anime

In [ ]:
ratings_por_anime = ratings.groupby('anime_id')['rating'].count()

print('Estadísticas de ratings por anime:')
print(ratings_por_anime.describe().apply(lambda x: f'{x:,.1f}'))
print(f'\nAnimes con al menos 1 rating: {ratings["anime_id"].nunique():,} de {anime["anime_id"].nunique():,} total')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución log
np.log1p(ratings_por_anime).hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de ratings por anime (log scale)')
axes[0].set_xlabel('log(1 + cantidad de ratings)')
axes[0].set_ylabel('Cantidad de animes')

# Top 15 animes más rateados
top_rateados = ratings_por_anime.nlargest(15).reset_index()
top_rateados = top_rateados.merge(anime[['anime_id', 'name']], on='anime_id')
sns.barplot(x='rating', y='name', data=top_rateados, ax=axes[1], palette='Blues_r')
axes[1].set_title('Top 15 animes con más ratings')
axes[1].set_xlabel('Cantidad de ratings')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### 4.4 Sparsity de la matriz usuario-anime

La sparsity es clave para sistemas de recomendación: indica qué tan "vacía" está la matriz de interacciones.

In [ ]:
n_usuarios = ratings['user_id'].nunique()
n_animes = ratings['anime_id'].nunique()
n_interacciones = len(ratings)
n_posibles = n_usuarios * n_animes
sparsity = 1 - (n_interacciones / n_posibles)

print(f'Usuarios únicos:           {n_usuarios:,}')
print(f'Animes únicos:             {n_animes:,}')
print(f'Interacciones totales:     {n_interacciones:,}')
print(f'Combinaciones posibles:    {n_posibles:,}')
print(f'Sparsity:                  {sparsity*100:.3f}%')
print(f'Densidad:                  {(1-sparsity)*100:.3f}%')

---
## 5. Resumen de hallazgos

### Dataset de Animes
- El formato dominante es **TV** seguido de **OVA** y **Movie**.
- Los géneros más comunes son **Comedy**, **Action** y **Adventure**.
- El rating promedio de los animes ronda el **6.5**, con distribución levemente sesgada a la izquierda.
- La popularidad (miembros) tiene una distribución muy sesgada: pocos animes concentran la mayoría de los seguidores.

### Dataset de Ratings
- El **38%** aprox. de las interacciones son implícitas (rating = -1), lo que representa usuarios que vieron el anime sin calificarlo.
- La distribución de ratings explícitos está sesgada hacia valores altos (7-8), típico de datasets de entretenimiento.
- Hay un importante problema de **cold-start**: muchos usuarios tienen muy pocos ratings.
- La matriz usuario-anime tiene una **sparsity muy alta** (>99%), lo que es el escenario clásico para Collaborative Filtering.

### Implicancias para el Sistema de Recomendación
- La alta sparsity favorece técnicas como **Matrix Factorization (SVD)** o **KNN Colaborativo**.
- Los ratings implícitos (-1) pueden aprovecharse como señal adicional.
- Los géneros son útiles para un enfoque **Content-Based** como complemento.
- Habrá que decidir cómo tratar usuarios con pocos ratings (cold-start problem).